In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score

# Load data
df = pd.read_csv("datasets/rank predictor.csv").dropna()

# Features
df['pct_score'] = df['Marks'] / df['max marks']
df['pct2'] = df['pct_score'] ** 2
df['pct3'] = df['pct_score'] ** 3

X = df[['pct_score', 'pct2', 'pct3', 'no of candidates', 'max marks']]
y = np.log(df['Rank'])

# Train
model = GradientBoostingRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, subsample=0.8, random_state=42)
model.fit(X, y)

# Predict function
def predict_rank(marks, max_marks, candidates):
    pct = marks / max_marks
    X_pred = pd.DataFrame([[pct, pct**2, pct**3, candidates, max_marks]],
                           columns=['pct_score', 'pct2', 'pct3', 'no of candidates', 'max marks'])
    return int(np.round(np.exp(model.predict(X_pred)[0])))
import joblib

# Model ko file mein save karein
joblib.dump(model, "rank_model.pkl")
# Test it
print(predict_rank(marks=352, max_marks=372, candidates=147678))

1


In [25]:
print(predict_rank(marks=50, max_marks=360, candidates=180000))

37975


In [30]:
import pandas as pd
import numpy as np

cutoff_df = pd.read_csv("examlevel.csv")

def check_qualify(overall, math, physics, chem, total_marks):
    # normalize to percentage for fair comparison across years
    pct_overall = overall / total_marks
    pct_math    = math / total_marks
    pct_physics = physics / total_marks
    pct_chem    = chem / total_marks

    qualified_years = 0
    for _, row in cutoff_df.iterrows():
        yr_total = row['Total_Marks_Paper']
        if (pct_overall >= row['GE_Aggregate_Cutoff'] / yr_total and
            pct_math    >= row['Cutoff_Math']         / yr_total and
            pct_physics >= row['Cutoff_Phys']         / yr_total and
            pct_chem    >= row['Cutoff_Chem']         / yr_total):
            qualified_years += 1

    rate = qualified_years / len(cutoff_df)

    if rate >= 0.7:
        verdict = "LIKELY TO QUALIFY"
    elif rate >= 0.4:
        verdict = "BORDERLINE"
    else:
        verdict = "UNLIKELY TO QUALIFY"

    print(f"Verdict          : {verdict}")
    print(f"Would qualify in : {qualified_years}/{len(cutoff_df)} past years ({rate*100:.1f}%)")
    return verdict

# --- Input ---
check_qualify(
    overall     = 150,
    math        = 50,
    physics     = 50,
    chem        = 50,
    total_marks = 360
)

Verdict          : LIKELY TO QUALIFY
Would qualify in : 14/15 past years (93.3%)


'LIKELY TO QUALIFY'

In [2]:
import sys
import sklearn

print("Python:", sys.version)
print("Python path:", sys.executable)
print("sklearn:", sklearn.__version__)

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Python path: c:\Users\udayv\anaconda3\python.exe
sklearn: 1.7.2
